# Linear Reduced GP: End-to-End High-Dimensional BO

PCA / PLS / Random Projectionによるsurrogate-side input reductionと、candidate searchの座標系を区別します。このNotebookでは`PCAGP`を代表として実際のcandidate generationまで実行します。

関連理論:

- [Dimensionality Reduction GP](../../docs/theory/18_dimensionality_reduction_gp.md)
- [High-dimensional Search Strategies](../../docs/theory/20_high_dimensional_search.md)

```text
original/public X (10D)
        ↓
    PCAGP reducer
        ↓
model latent Z_m (3D)
        ↓
   GP posterior
        ↓
 acquisition(X)
        ↓
optimize in original/public X (10D)
        ↓
 candidate X (10D)
```

重要なのは、model latent `Z_m`とcandidate-search coordinateを同一視しないことです。

## このNotebookの到達目標

1. 高次元の**元の入力空間**と、GP内部の**縮約表現**を区別する
2. PCA・PLS・Random Projectionの仮定を理解し、同じデータで比較する
3. `model.fit()` による学習と、元の入力座標での `posterior()` を確認する
4. qLogEIと `optimize_acqf()` で候補を提案し、観測・再学習まで体験する
5. Mixed・MultiTask・Kroneckerでは、出力形状と候補探索方法が異なることを理解する

**重要：** このNotebookは既存のrobotorchanモデルとBoTorchの公開APIを再利用します。全モデルを同一の最適化手法に統一することは目的としません。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import PCAGP, PLSGP, RandomProjectionGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Sparse high-dimensional objective

10次元入力ですが、目的関数へ強く効くのは一部のdimensionだけにします。

In [ ]:
input_dim = 10
n_train = 28
bounds = torch.stack(
    [torch.zeros(input_dim), torch.ones(input_dim)],
    dim=0,
)


def objective(X: torch.Tensor) -> torch.Tensor:
    value = (
        torch.sin(2 * torch.pi * X[..., 0])
        + 0.6 * (X[..., 3] - 0.3) ** 2
        - 0.5 * X[..., 7]
    )
    return value.unsqueeze(-1)


train_X = torch.rand(n_train, input_dim)
train_Y = objective(train_X)
print(train_X.shape, train_Y.shape)

### 3つの次元削減の使い分け

| 手法 | 縮約軸の決め方 | 注意点 |
|---|---|---|
| PCA | 入力Xの分散が大きい方向 | 目的Yに重要でもXの分散が小さい方向を落とす可能性 |
| PLS | 入力Xと目的Yの関係を利用 | データが少ないと方向の推定が不安定になり得る |
| Random Projection | ランダムな線形写像 | Yの情報を使って重要方向を学習するわけではない |

**入力を3次元に縮約しても、獲得関数最適化の探索変数は元の10次元のままです。** これはREMBOのように探索自体を低次元化する方法とは異なります。

## 3. Compare linear reducers

すべてのmodelへ渡すtraining inputはoriginal 10Dです。reducerだけがmodel内部の表現を変えます。

In [ ]:
models = {
    "PCA": PCAGP(train_X, train_Y, n_components=3),
    "PLS": PLSGP(train_X, train_Y, n_components=3),
    "Random": RandomProjectionGP(
        train_X,
        train_Y,
        n_components=3,
        random_state=0,
    ),
}

for name, model in models.items():
    print(
        name,
        "public=", model.original_input_dim,
        "latent=", model.reduced_input_dim,
    )

## 4. Fit all linear reducers

PCA・PLS・Random Projectionの各モデルを既存の`make_mll()`で学習します。


In [ ]:
for name, model in models.items():
    model.fit()
    model.eval()
    print(
        name,
        "raw_train_X:", model.raw_train_X.shape,
        "public dim:", model.original_input_dim,
        "latent dim:", model.reduced_input_dim,
    )

model = models["PCA"]


## 5. Posterior and sampling in public coordinates

callerはlatent coordinateを作らず、original-space Xを`posterior()`へ渡します。

In [ ]:
test_X = torch.rand(4, input_dim)
with torch.no_grad():
    posterior = model.posterior(test_X)
    samples = posterior.rsample(torch.Size([16]))

print("test_X:", test_X.shape)
print("posterior mean:", posterior.mean.shape)
print("samples:", samples.shape)

### 予測平均と不確実性を1変数方向で可視化

10次元全体を直接描くことはできないため、ここでは1列目の入力だけを動かし、残りの9列を0.5に固定した**断面**を表示します。PCA・PLS・Random Projectionの予測を比較します。

帯は潜在関数の事後標準偏差の約1.96倍であり、観測ノイズを含む予測区間ではありません。**断面の可視化は10次元全域の精度を保証しません。**

In [ ]:
import matplotlib.pyplot as plt

slice_X = torch.full((121, input_dim), 0.5)
slice_X[:, 0] = torch.linspace(0.0, 1.0, len(slice_X))
true_slice_Y = objective(slice_X).squeeze(-1)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, surrogate) in zip(axes, models.items()):
    surrogate.eval()
    with torch.no_grad():
        slice_post = surrogate.posterior(slice_X)
        mu = slice_post.mean.squeeze(-1)
        sigma = slice_post.variance.clamp_min(0).sqrt().squeeze(-1)
    ax.plot(slice_X[:, 0], true_slice_Y, "--", label="true slice")
    ax.plot(slice_X[:, 0], mu, label="posterior mean")
    ax.fill_between(slice_X[:, 0], mu - 1.96 * sigma,
                    mu + 1.96 * sigma, alpha=0.2,
                    label="approx. 95% latent interval")
    ax.set_title(name)
    ax.set_xlabel("x[0] (other dimensions fixed at 0.5)")
axes[0].set_ylabel("objective")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Acquisition function

MC acquisitionもoriginal/public Xを受け取ります。内部reductionはmodelの責務です。

In [ ]:
acqf = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
)
probe_X = torch.full((1, 1, input_dim), 0.5)
print("acquisition probe:", acqf(probe_X))

### 獲得関数の役割

事後平均最大化は既知の高い値を優先します。一方、qLogEIは現在の最良観測値 `best_f` を改善する可能性を、予測分布を使って評価します。

`qLogExpectedImprovement` は獲得関数の**計算**を担当し、`optimize_acqf` はその獲得関数を**最大化する入力候補の探索**を担当します。どちらも10次元の元の入力座標を受け取ります。

## 7. Candidate generation in original space

PCAが3次元だからといって`optimize_acqf`へ3次元boundsを渡しません。このworkflowではcandidate searchはoriginal 10D boxで行われ、返るcandidateも10Dです。

In [ ]:
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([1, input_dim])
assert torch.all(candidate >= bounds[0])
assert torch.all(candidate <= bounds[1])
print("candidate shape:", candidate.shape)
print("acquisition value:", acq_value)

## 7b. Candidate proposals for PLS and Random Projection

各モデルの獲得関数を元の10次元空間で最適化し、候補を評価します。


In [ ]:
for name in ("PLS", "Random"):
    other_model = models[name]
    other_acqf = qLogExpectedImprovement(
        model=other_model,
        best_f=train_Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    other_candidate, other_value = optimize_acqf(
        acq_function=other_acqf,
        bounds=bounds,
        q=1,
        num_restarts=2,
        raw_samples=32,
    )
    assert other_candidate.shape == torch.Size([1, input_dim])
    print(name, "candidate:", other_candidate)
    print(name, "acquisition:", other_value)
    print(name, "objective:", objective(other_candidate))


## 7c. Mixed-variable reduced GPs

連続10次元とカテゴリ1次元を入力とし、次元削減は連続成分だけに適用します。カテゴリ値は有限候補プールで保持し、各モデルの獲得関数から次の候補を選びます。


In [ ]:
from robotorchan.models import MixedPCAGP, MixedPLSGP, MixedRandomProjectionGP

mixed_train_X = torch.cat(
    [train_X, torch.randint(0, 2, (n_train, 1)).to(train_X)],
    dim=-1,
)


def mixed_objective(X: torch.Tensor) -> torch.Tensor:
    return objective(X[..., :input_dim]) + 0.2 * X[..., -1:]


mixed_train_Y = mixed_objective(mixed_train_X)
mixed_models = {
    "Mixed PCA": MixedPCAGP(
        mixed_train_X, mixed_train_Y, n_components=3, cat_dims=[input_dim]
    ),
    "Mixed PLS": MixedPLSGP(
        mixed_train_X, mixed_train_Y, n_components=3, cat_dims=[input_dim]
    ),
    "Mixed Random": MixedRandomProjectionGP(
        mixed_train_X,
        mixed_train_Y,
        n_components=3,
        cat_dims=[input_dim],
        random_state=0,
    ),
}
mixed_pool = torch.cat(
    [
        torch.rand(64, input_dim),
        torch.randint(0, 2, (64, 1)).to(train_X),
    ],
    dim=-1,
)
for name, mixed_model in mixed_models.items():
    mixed_model.fit()
    mixed_model.eval()
    mixed_acqf = qLogExpectedImprovement(
        model=mixed_model,
        best_f=mixed_train_Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    with torch.no_grad():
        mixed_values = mixed_acqf(mixed_pool.unsqueeze(-2)).reshape(-1)
    mixed_candidate = mixed_pool[mixed_values.argmax()].unsqueeze(0)
    assert mixed_candidate.shape == (1, input_dim + 1)
    assert mixed_candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", mixed_candidate)
    print(name, "objective:", mixed_objective(mixed_candidate))


### Mixedモデルの候補選択について

Mixedモデルはカテゴリ値が離散的なので、連続変数と同じようにすべての列を自由に動かすことはできません。この例では64個の有効なMixed候補を生成し、**qLogEIの値で順位付け**しています。

これは事後平均のみの選択ではありません。一方、`optimize_acqf_mixed()` による連続部分の最適化とも異なり、候補集合に含まれない点は選べません。

## 7d. Generic ReducedGP with an explicit reducer

専用のPCAGPだけでなく、汎用のReducedGPへPCAInputReducerを渡す場合も候補提案まで確認します。


In [ ]:
from robotorchan.models import ReducedGP
from robotorchan.reduction import PCAInputReducer

generic_model = ReducedGP(
    train_X,
    train_Y,
    input_reducer=PCAInputReducer(n_components=3),
)
generic_model.fit()
generic_model.eval()
generic_acqf = qLogExpectedImprovement(
    model=generic_model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
)
generic_pool = torch.rand(64, input_dim)
with torch.no_grad():
    generic_values = generic_acqf(generic_pool.unsqueeze(-2)).reshape(-1)
generic_candidate = generic_pool[generic_values.argmax()].unsqueeze(0)
assert generic_candidate.shape == (1, input_dim)
print("ReducedGP candidate:", generic_candidate)
print("ReducedGP objective:", objective(generic_candidate))


## 7e. Long-format PCA MultiTaskGP: task-specific proposal

タスク識別子は次元削減せず、目的タスク0の候補を元の入力座標で提案します。ここではposterior meanに基づく有限候補探索を使用します。


In [ ]:
from robotorchan.models import PCAMultiTaskGP

task_train_X = torch.cat(
    [
        torch.cat([train_X, torch.zeros(n_train, 1)], dim=-1),
        torch.cat([train_X, torch.ones(n_train, 1)], dim=-1),
    ],
    dim=0,
)
task_train_Y = torch.cat([train_Y, train_Y + 0.15], dim=0)
task_model = PCAMultiTaskGP(
    task_train_X,
    task_train_Y,
    task_feature=input_dim,
    n_components=3,
)
task_model.fit()
task_model.eval()
task_pool = torch.cat(
    [torch.rand(64, input_dim), torch.zeros(64, 1)], dim=-1
)
with torch.no_grad():
    task_scores = task_model.posterior(task_pool).mean.reshape(-1)
task_candidate = task_pool[task_scores.argmax()].unsqueeze(0)
assert task_candidate.shape == (1, input_dim + 1)
assert task_candidate[0, -1].item() == 0
print("PCA MultiTask candidate:", task_candidate)
print("Task 0 objective:", objective(task_candidate[:, :input_dim]))


### MultiTaskの候補選択について

long-formatではタスクIDを入力に持ち、block-design Kroneckerでは複数タスクが出力列に並びます。どちらも対象タスクを決めてから候補を比較する必要があります。

このNotebookのMultiTask系は**有限候補集合の事後平均**を利用します。これは標準獲得関数が使用不可能という判定ではなく、タスク固定・出力選択を伴う最適化をここでは追加していない、という意味です。

## 7f. PLS and Random Projection MultiTask proposals

同じlong-format学習データを用い、目的タスク0のposterior meanで有限候補プールを順位付けします。


In [ ]:
from robotorchan.models import PLSMultiTaskGP, RandomProjectionMultiTaskGP

task_variants = {
    "PLS MultiTask": PLSMultiTaskGP(
        task_train_X, task_train_Y, task_feature=input_dim, n_components=3
    ),
    "Random MultiTask": RandomProjectionMultiTaskGP(
        task_train_X,
        task_train_Y,
        task_feature=input_dim,
        n_components=3,
        random_state=0,
    ),
}
for name, variant in task_variants.items():
    variant.fit()
    variant.eval()
    with torch.no_grad():
        scores = variant.posterior(task_pool).mean.reshape(-1)
    proposal = task_pool[scores.argmax()].unsqueeze(0)
    assert proposal.shape == (1, input_dim + 1)
    assert proposal[0, -1].item() == 0
    print(name, "candidate:", proposal)
    print(name, "objective:", objective(proposal[:, :input_dim]))


## 7g. Block-design Kronecker MultiTask candidate proposal

同一の入力点で2タスクを観測するblock-designデータを使います。Posteriorのタスク0の予測平均に基づいて候補を選び、元の10次元座標で評価します。


In [ ]:
from robotorchan.models import PCAKroneckerMultiTaskGP

kronecker_train_Y = torch.cat([train_Y, train_Y + 0.15], dim=-1)
kronecker_model = PCAKroneckerMultiTaskGP(
    train_X,
    kronecker_train_Y,
    n_components=3,
)
kronecker_model.fit()
kronecker_model.eval()
kronecker_pool = torch.rand(64, input_dim)
with torch.no_grad():
    kronecker_posterior = kronecker_model.posterior(kronecker_pool)
    kronecker_scores = kronecker_posterior.mean[..., 0].reshape(-1)
kronecker_candidate = kronecker_pool[kronecker_scores.argmax()].unsqueeze(0)
assert kronecker_candidate.shape == (1, input_dim)
print("PCA Kronecker candidate:", kronecker_candidate)
print("Task 0 objective:", objective(kronecker_candidate))


## 7h. PLS and Random Projection Kronecker proposals

Block-design形式の2タスク出力を使用し、各モデルのタスク0の予測平均に基づいて元の入力空間で候補を選びます。


In [ ]:
from robotorchan.models import (
    PLSKroneckerMultiTaskGP,
    RandomProjectionKroneckerMultiTaskGP,
)

kronecker_variants = {
    "PLS Kronecker": PLSKroneckerMultiTaskGP(
        train_X, kronecker_train_Y, n_components=3
    ),
    "Random Kronecker": RandomProjectionKroneckerMultiTaskGP(
        train_X, kronecker_train_Y, n_components=3, random_state=0
    ),
}
for name, variant in kronecker_variants.items():
    variant.fit()
    variant.eval()
    with torch.no_grad():
        posterior = variant.posterior(kronecker_pool)
        scores = posterior.mean[..., 0].reshape(-1)
    proposal = kronecker_pool[scores.argmax()].unsqueeze(0)
    assert proposal.shape == (1, input_dim)
    print(name, "candidate:", proposal)
    print(name, "task 0 objective:", objective(proposal))


## 7i. Mixed Reduced MultiTask candidate proposal

連続特徴量・カテゴリ特徴量・タスクIDを別々に扱い、PCAによる削減は連続特徴量だけに適用します。目的タスク0について元の入力座標で候補を提案します。


In [ ]:
from robotorchan.models import MixedReducedMultiTaskGP

mixed_task_X = torch.cat(
    [
        torch.cat(
            [mixed_train_X, torch.zeros(n_train, 1)],
            dim=-1,
        ),
        torch.cat(
            [mixed_train_X, torch.ones(n_train, 1)],
            dim=-1,
        ),
    ],
    dim=0,
)
mixed_task_Y = torch.cat(
    [mixed_train_Y, mixed_train_Y + 0.15], dim=0
)
mixed_task_model = MixedReducedMultiTaskGP(
    mixed_task_X,
    mixed_task_Y,
    task_feature=input_dim + 1,
    cat_dims=[input_dim],
    input_reducer=PCAInputReducer(n_components=3),
)
mixed_task_model.fit()
mixed_task_model.eval()
mixed_task_pool = torch.cat(
    [mixed_pool, torch.zeros(mixed_pool.shape[0], 1)], dim=-1
)
with torch.no_grad():
    mixed_task_scores = mixed_task_model.posterior(
        mixed_task_pool
    ).mean.reshape(-1)
mixed_task_candidate = mixed_task_pool[
    mixed_task_scores.argmax()
].unsqueeze(0)
assert mixed_task_candidate.shape == (1, input_dim + 2)
assert mixed_task_candidate[0, -1].item() == 0
assert mixed_task_candidate[0, -2].item() in (0, 1)
print("Mixed Reduced MultiTask candidate:", mixed_task_candidate)
print(
    "Task 0 objective:",
    mixed_objective(mixed_task_candidate[:, : input_dim + 1]),
)


## 7j. Mixed Reduced Kronecker MultiTask proposal

カテゴリ特徴量を保持したblock-design形式の2タスク出力を学習します。タスク0の予測平均で有限候補プールを順位付けします。


In [ ]:
from robotorchan.models import MixedReducedKroneckerMultiTaskGP

mixed_kronecker_Y = torch.cat(
    [mixed_train_Y, mixed_train_Y + 0.15], dim=-1
)
mixed_kronecker_model = MixedReducedKroneckerMultiTaskGP(
    mixed_train_X,
    mixed_kronecker_Y,
    cat_dims=[input_dim],
    input_reducer=PCAInputReducer(n_components=3),
)
mixed_kronecker_model.fit()
mixed_kronecker_model.eval()
with torch.no_grad():
    mixed_kronecker_posterior = mixed_kronecker_model.posterior(mixed_pool)
    mixed_kronecker_scores = mixed_kronecker_posterior.mean[..., 0].reshape(-1)
mixed_kronecker_candidate = mixed_pool[
    mixed_kronecker_scores.argmax()
].unsqueeze(0)
assert mixed_kronecker_candidate.shape == (1, input_dim + 1)
assert mixed_kronecker_candidate[0, -1].item() in (0, 1)
print("Mixed Reduced Kronecker candidate:", mixed_kronecker_candidate)
print("Task 0 objective:", mixed_objective(mixed_kronecker_candidate))


## 7k. Generic reduced model variants

専用PCAクラスと同じ設定を、汎用Mixed / MultiTask / Kroneckerクラスへ明示的な入力reducerとして渡します。候補提案は元の入力座標の有限候補プールで行います。


In [ ]:
from robotorchan.models import (
    MixedReducedGP,
    ReducedKroneckerMultiTaskGP,
    ReducedMultiTaskGP,
)

generic_variants = (
    (
        "MixedReducedGP",
        MixedReducedGP(
            mixed_train_X,
            mixed_train_Y,
            input_reducer=PCAInputReducer(n_components=3),
            cat_dims=[input_dim],
        ),
        mixed_pool,
        mixed_objective,
        None,
    ),
    (
        "ReducedMultiTaskGP",
        ReducedMultiTaskGP(
            task_train_X,
            task_train_Y,
            task_feature=input_dim,
            input_reducer=PCAInputReducer(n_components=3),
        ),
        task_pool,
        objective,
        input_dim,
    ),
    (
        "ReducedKroneckerMultiTaskGP",
        ReducedKroneckerMultiTaskGP(
            train_X,
            kronecker_train_Y,
            input_reducer=PCAInputReducer(n_components=3),
        ),
        kronecker_pool,
        objective,
        0,
    ),
)
for name, generic_variant, pool, evaluate, task_index in generic_variants:
    generic_variant.fit()
    generic_variant.eval()
    with torch.no_grad():
        means = generic_variant.posterior(pool).mean
        scores = (
            means.reshape(-1)
            if task_index is None or task_index == input_dim
            else means[..., task_index].reshape(-1)
        )
    proposal = pool[scores.argmax()].unsqueeze(0)
    if task_index == input_dim:
        value = evaluate(proposal[:, :input_dim])
    else:
        value = evaluate(proposal)
    print(name, "candidate:", proposal)
    print(name, "objective:", value)


## 7l. Output reduction: PCA and PLS

入力次元ではなく複数の出力を削減します。元の出力空間へ復元されたPosteriorの第1出力を目的とし、有限候補プールから次の評価点を選択します。


In [ ]:
from robotorchan.models import OutputPCAGP, OutputPLSGP

output_train_Y = torch.cat(
    [
        train_Y,
        0.7 * train_Y + 0.2 * train_X[:, 1:2],
        -0.4 * train_Y + 0.3 * train_X[:, 2:3],
    ],
    dim=-1,
)
output_models = {
    "Output PCA": OutputPCAGP(train_X, output_train_Y, n_components=2),
    "Output PLS": OutputPLSGP(train_X, output_train_Y, n_components=2),
}
output_pool = torch.rand(64, input_dim)
for name, output_model in output_models.items():
    output_model.fit()
    output_model.eval()
    with torch.no_grad():
        output_posterior = output_model.posterior(output_pool)
        assert output_posterior.mean.shape == (64, 3)
        output_scores = output_posterior.mean[:, 0]
    output_candidate = output_pool[output_scores.argmax()].unsqueeze(0)
    assert output_candidate.shape == (1, input_dim)
    print(name, "candidate:", output_candidate)
    print(name, "first output objective:", objective(output_candidate))


## 8. Candidate evaluation and update

In [ ]:
candidate_Y = objective(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("candidate objective:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

### 候補評価と再学習の意味

提案した `candidate` を目的関数で評価し、元の10次元座標のまま `updated_X` に追加します。次の学習ではPCAの縮約表現も新しい観測集合から再構築します。

ここでの再学習は**1回分の更新例**です。獲得関数の最適化を繰り返す本格的なBOループや性能ベンチマークとは区別してください。

## 9. One refit step

次のiterationではoriginal-space observationを追加してreducer/modelを再構築します。latent coordinateをtraining dataとして保存しません。

In [ ]:
next_model = PCAGP(updated_X, updated_Y, n_components=3)
next_model.fit()
next_model.eval()
next_acqf = qLogExpectedImprovement(
    model=next_model,
    best_f=updated_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
)
next_candidate, _ = optimize_acqf(
    acq_function=next_acqf,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

assert next_candidate.shape[-1] == input_dim
print("next candidate:", next_candidate)

## 10. Surrogate reduction is not search reduction

```text
PCAGP / PLSGP / RandomProjectionGP
    reduce model representation
    public candidate remains original X

REMBO / HeSBO / ALEBO / BAxUS
    change candidate-search geometry
    search target Z_s is a separate coordinate system

TuRBO
    keeps original coordinates but localizes candidate search
```

したがってPCAを使っただけで高次元`optimize_acqf`の難しさまで解消したとは言えません。

## 11. Other high-dimensional surrogate families

- Fully Bayesian SAAS: sparse relevanceをNUTSで推論します。`08_saas_gp.ipynb`で学習contractを扱います。
- MAP-SAAS / additive GP: より軽量なexact-GP系です。`09_map_saas_and_additive_gp.ipynb`で比較します。
- Neural reduction: nonlinear representationを`20_neural_reduction_gp.ipynb`で扱います。
- `23_high_dimensional_bo_benchmark.ipynb`: 複数seedの有限candidate-pool benchmarkでsurrogate差を比較します。

これらを1つの巨大Notebookへ統合せず、ここではpublic-space candidate contractを代表的に実証します。

## 12. When to use and caveats

- PCAはXの分散方向を使うため、目的Yに重要な低分散方向を落とす可能性があります。
- PLSはsupervisedですが、少数データではprojection推定自体が不安定になり得ます。
- Random Projectionはdata-driven reducerではなく、固定されたランダム写像です。
- SAASは「少数dimensionが重要」というsparsity仮定を利用し、input reductionとは別アプローチです。
- 高次元でcandidate optimizationがボトルネックなら、surrogate変更だけでなくTuRBO / REMBO / ALEBO / BAxUS等のsearch strategyを検討します。
- model latent `Z_m`とsearch target `Z_s`を混同しません。

## 13. References

高次元モデルの仮定とsearch strategyの違いは冒頭のTheory章を参照してください。TuRBO / Trust Regionは後続PhaseでEnd-to-End workflowとして扱います。